In [1]:
from SPARQLWrapper import SPARQLWrapper, TURTLE, JSON, CSV
import subprocess
import time
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import glob
from itertools import combinations
import itertools
from utility import *
import csv
from collections import defaultdict
import ast
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
current_dir = os.getcwd()
MICA_MATRICES_REACTOME = os.path.join(current_dir, "../../Results/PathwayComembership/05_MatricesReactome/*.csv")
NSP_INTER_TOP_PATHWAYS = os.path.join(current_dir, "../../Results/PathwayComembership/06_ScalingReactome/NSPInterTop.csv")
REACTOME_ER_PER_PATHWAY = os.path.join(current_dir, "../../Results/UtilityFiles/Reactome96_UpPerPathway.csv")
BIOPAX_ONTOLOGY = os.path.join(current_dir, "../../Data/BioPAX/BioPAXOntology/biopax-level3.owl")
REACTOME_BIOPAX = os.path.join(current_dir, "../../Data/BioPAX/ReactomeBioPAX/Homo_sapiens_v96.owl")
WEIGHTED_TOP_PATHWAY_CLIQUES = os.path.join(current_dir, "../../Results/PathwayComembership/02_WeightedComembershipCliques/")
TGF_BETA_FOLDER = os.path.join(current_dir, "../../Results/PathwayComembership/07_TGFbeta/")

prefixes = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>
"""

In [3]:
def load_entity_refs_per_pathway(file) -> pd.DataFrame:
    """Load the (pathway, UniProt ID) table for this pathway file."""
    return pd.read_csv(
        file,
        sep=",", header=0,
    )

def build_entity_refs_per_pathway_dict(er_per_pathway: pd.DataFrame) -> dict[str, list[str]]:
    """Build {pathway: [uniprot_id, ...]} from the raw (pathway, id) table."""
    dico: dict[str, list[str]] = {}
    for pathway, entity_id in er_per_pathway.iloc[:, [0, 1]].values:
        dico.setdefault(pathway, []).append(entity_id)
    return dico

def compute_similarity(Pathway1, Pathway2, dicoErperPathway, nbERTotal):
    if Pathway1 in dicoErperPathway.keys() and Pathway2 in dicoErperPathway.keys():
        ER_pathway1 = dicoErperPathway[Pathway1]
        ER_pathway2 = dicoErperPathway[Pathway2]
        nb_ER_MICA = (len(set(ER_pathway1)) + len(set(ER_pathway2))) - len(list(set(ER_pathway1) & set(ER_pathway2)))
    else:
        nb_ER_MICA = 0
    if nb_ER_MICA != 0 and nb_ER_MICA != nbERTotal:
        ic = -math.log10(nb_ER_MICA/nbERTotal)
    else:
        ic = 0
    return ic

def score_pair(self, entity1: str, entity2: str, matrix_mica: dict, matrix_nsp: dict, matrix_mica_reactome:dict) -> tuple[float, str] | None:
        """
        Return (score, provenance) for a pair of co-membered entities, or
        None if either entity has no known pathway parent.
        """
        if entity1 not in self.dico_most_precise_parent or entity2 not in self.dico_most_precise_parent:
            return None

        parents1, ancestor_pathways1 = self._get_parents_and_ancestor_pathways(entity1)
        parents2, ancestor_pathways2 = self._get_parents_and_ancestor_pathways(entity2)

        mica, mica_score, mica_score_reactome = self._get_mica_score(parents1, parents2, nb_er_reactome)
        next_step_score = self._get_max_next_step_score(ancestor_pathways1, ancestor_pathways2)
        matrix_mica[(entity1, entity2)] = mica_score
        matrix_mica[(entity2, entity1)] = mica_score
        matrix_mica_reactome[(entity1, entity2)] = mica_score_reactome
        matrix_mica_reactome[(entity2, entity1)] = mica_score_reactome
        matrix_nsp[(entity1, entity2)] = next_step_score
        matrix_nsp[(entity2, entity1)] = next_step_score

        if mica_score >= next_step_score:
            return mica_score, "scoreMICA"
        return next_step_score, "scoreNextStep"

dico_top_pathway_ids = {
    "01": "Autophagy",
    "02": "CellCycle",
    "03": "CellCellCommunication",
    "04": "CellularResponseToStimuli",
    "05": "ChromatinOrganization",
    "06": "CircadianClock",
    "07": "DevelopmentalBiology",
    "08": "DigestionAndAbsorption",
    "09": "Disease",
    "10": "DNARepair",
    "11": "DNAReplication",
    "12": "DrugADME",
    "13": "ExtracellularMatrixOrganization",
    "14": "GeneExpression(Transcription)",
    "15": "Hemostasis",
    "16": "ImmuneSystem",
    "17": "Metabolism",
    "18": "MetabolismOfProteins",
    "19": "MetabolismOfRNA",
    "20": "MuscleContraction",
    "21": "NeuronalSystem",
    "22": "OrganelleBiogenesisAndMaintenance",
    "23": "ProgrammedCellDeath",
    "24": "ProteinLocalization",
    "25": "Reproduction",
    "26": "SensoryPerception",
    "27": "SignalTransduction",
    "28": "TransportOfSmallMolecules",
    "29": "VesicleMediatedTransport"
}

##### 1.2 - Create all possible pairs of Reactome proteins to create the comembership clique

In [4]:
reactome_proteins = os.path.join(current_dir, f"../../Results/PathwayComembership/06_ScalingReactome/Reactome_ProteinList.csv")
df = pd.read_csv(reactome_proteins, sep=",", header=0)
proteins = df['id1'].tolist()
pairs = list(combinations(proteins, 2))
df_pairs = pd.DataFrame(pairs, columns=['id1', 'id2'])
output_file = os.path.join(current_dir, f"../../Results/PathwayComembership/06_ScalingReactome/Reactome_ComembershipClique.csv")
df_pairs.to_csv(output_file, index=False)
print(len(df_pairs))

73223151


In [5]:
er_per_pathway = load_entity_refs_per_pathway(REACTOME_ER_PER_PATHWAY)
dico_er_per_pathway = build_entity_refs_per_pathway_dict(er_per_pathway)
print(dico_er_per_pathway)

dico_parents_up = dict()
for index, row in er_per_pathway.iterrows():
    if not row[1] in dico_parents_up.keys():
        dico_parents_up[row[1]] = [row[0]]
    else:
        dico_parents_up[row[1]] += [row[0]]
#print(dico_parents_up)
print(len(dico_parents_up))

{'R-HSA-5357609': ['P46976', 'P10253'], 'R-HSA-933543': ['Q13158', 'Q14790', 'Q92851', 'Q13546', 'Q7Z434', 'Q9BYX4', 'O95786', 'Q9C037', 'Q14258', 'Q8IUD6', 'O14920', 'Q9Y6K9', 'O15111'], 'R-HSA-72312': ['Q9H633', 'Q8TD47', 'P22090', 'P62701', 'Q9BUL9', 'P62841', 'P62244', 'P62263', 'P15880', 'Q96GA3', 'P46782', 'P61247', 'O95059', 'P62269', 'P62241', 'P62861', 'P62847', 'P78346', 'P62854', 'Q13895', 'P62277', 'Q9BVS4', 'P39019', 'P62249', 'P42677', 'Q71UM5', 'P62081', 'P78345', 'Q9BRS2', 'P08865', 'P62851', 'P62273', 'P62753', 'P25398', 'P46781', 'P08708', 'P62280', 'P63220', 'Q9ULX3', 'Q9NRX1', 'P23396', 'P62857', 'P48730', 'P49674', 'O75818', 'Q2NL82', 'P46783', 'P62979', 'P62266', 'P60866', 'O14730', 'O00567', 'P22087', 'Q9Y2X3', 'P55769', 'O43818', 'Q68CQ4', 'O75691', 'P78316', 'Q9BVI4', 'Q92979', 'Q14690', 'Q9NQZ2', 'Q14692', 'Q9Y2P8', 'Q9Y324', 'Q96G21', 'Q9NV31', 'O00566', 'Q969X6', 'Q15061', 'Q8IWA0', 'Q9H583', 'Q8TED0', 'Q13601', 'Q9H0S4', 'Q9Y3A2', 'Q9NV06', 'Q9H8H0', 'Q1278

/tmp/ipykernel_447429/1139861412.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if not row[1] in dico_parents_up.keys():
/tmp/ipykernel_447429/1139861412.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dico_parents_up[row[1]] = [row[0]]
/tmp/ipykernel_447429/1139861412.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dico_parents_up[row[1]] += [row[0]]


12087


#### Get pathway hierarchy of Reactome to link proteins to parent pathways

In [6]:
pathway_abstraction = pd.read_csv("../../Results/PathwayAbstraction/02_Reactome96Global/Reactome96_WeightedPathwayAbstraction.csv", sep=",", header=0)
G = create_networkx_graph(pathway_abstraction)
hierarchy_graph = extract_subgraph_by_interaction(G, "abstraction:IsAComponentOf")

list_pathways = list()
for index, row in pathway_abstraction.iterrows():
    pathway1 = row[0]
    pathway2 = row[2]
    if not pathway1 in list_pathways:
        list_pathways.append(pathway1)
    if not pathway2 in list_pathways:
        list_pathways.append(pathway2)
print(len(list_pathways))

dico_ancestors = dict()
for pathway in list_pathways:
    anc,_ = find_ancestors_and_descendants(hierarchy_graph, pathway)
    dico_ancestors[pathway] = anc
print(dico_ancestors)

Pathway abstraction loaded in networkx: 


/tmp/ipykernel_447429/3788782033.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pathway1 = row[0]
/tmp/ipykernel_447429/3788782033.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pathway2 = row[2]


2871
{'R-HSA-9612973': {'R-HSA-ROOT'}, 'R-HSA-ROOT': set(), 'R-HSA-1640170': {'R-HSA-ROOT'}, 'R-HSA-1500931': {'R-HSA-ROOT'}, 'R-HSA-8953897': {'R-HSA-ROOT'}, 'R-HSA-4839726': {'R-HSA-ROOT'}, 'R-HSA-9909396': {'R-HSA-ROOT'}, 'R-HSA-1266738': {'R-HSA-ROOT'}, 'R-HSA-8963743': {'R-HSA-ROOT'}, 'R-HSA-1643685': {'R-HSA-ROOT'}, 'R-HSA-73894': {'R-HSA-ROOT'}, 'R-HSA-69306': {'R-HSA-ROOT'}, 'R-HSA-9748784': {'R-HSA-ROOT'}, 'R-HSA-1474244': {'R-HSA-ROOT'}, 'R-HSA-74160': {'R-HSA-ROOT'}, 'R-HSA-109582': {'R-HSA-ROOT'}, 'R-HSA-168256': {'R-HSA-ROOT'}, 'R-HSA-1430728': {'R-HSA-ROOT'}, 'R-HSA-392499': {'R-HSA-ROOT'}, 'R-HSA-8953854': {'R-HSA-ROOT'}, 'R-HSA-397014': {'R-HSA-ROOT'}, 'R-HSA-112316': {'R-HSA-ROOT'}, 'R-HSA-1852241': {'R-HSA-ROOT'}, 'R-HSA-5357801': {'R-HSA-ROOT'}, 'R-HSA-9609507': {'R-HSA-ROOT'}, 'R-HSA-1474165': {'R-HSA-ROOT'}, 'R-HSA-9709957': {'R-HSA-ROOT'}, 'R-HSA-162582': {'R-HSA-ROOT'}, 'R-HSA-382551': {'R-HSA-ROOT'}, 'R-HSA-5653656': {'R-HSA-ROOT'}, 'R-HSA-9667769': {'R-HSA-9662

In [7]:
root = find_root(G, "abstraction:IsAComponentOf")
print(root)
# dico_most_precise_parent = dict()
# for prot, parents, in dico_parents_up.items():
#     max_shortest_path = 1#
#     if len(parents) == 1:
#         most_precise_parents.append(parents[0])
#     else:
#         shortest_paths = dict()
#         for parent in parents:
#             shortest_path_from_root = nx.shortest_path(hierarchy_graph.reverse(), root, parent)
#             shortest_paths[parent] = len(shortest_path_from_root)
#         most_precise_parents = [key for key, val in shortest_paths.items() if val == max(shortest_paths.values())]
#         dico_most_precise_parent[prot] = most_precise_parents
# print(dico_most_precise_parent)
# print(len(dico_most_precise_parent))

# df_parents = pd.DataFrame(columns=["prot", "parents"])
# i = 0
# for key, val in dico_most_precise_parent.items():
#     df_parents.at[i, "prot"] = key
#     df_parents.at[i, "parents"] = val
#     i += 1
# df_parents.to_csv("../../Results/PathwayComembership/06_ScalingReactome/ReactomeDicoParents.csv", sep=',', header=0, index=False)

R-HSA-ROOT


In [8]:
file_prot_parents = pd.read_csv("../../Results/PathwayComembership/06_ScalingReactome/ReactomeDicoParents.csv", sep=",", header=None)
dico_most_precise_parent = dict()
for index, row in file_prot_parents.iterrows():
    dico_most_precise_parent[row[0]] = ast.literal_eval(row[1])
print(dico_most_precise_parent)

{'P46976': ['R-HSA-5357609', 'R-HSA-3828062', 'R-HSA-3814836', 'R-HSA-3785653'], 'Q13158': ['R-HSA-2562578'], 'Q14790': ['R-HSA-2562578'], 'Q92851': ['R-HSA-6803207'], 'Q13546': ['R-HSA-2562578', 'R-HSA-937041'], 'Q7Z434': ['R-HSA-9705671', 'R-HSA-9692916'], 'Q9BYX4': ['R-HSA-9705671', 'R-HSA-9692916'], 'O95786': ['R-HSA-9705671', 'R-HSA-9692916'], 'Q9C037': ['R-HSA-9705671'], 'Q14258': ['R-HSA-9705671', 'R-HSA-9692916'], 'Q8IUD6': ['R-HSA-9705671'], 'O14920': ['R-HSA-9705671', 'R-HSA-975144', 'R-HSA-937041'], 'Q9Y6K9': ['R-HSA-9705671', 'R-HSA-975144', 'R-HSA-937041'], 'O15111': ['R-HSA-9705671', 'R-HSA-975144', 'R-HSA-937041'], 'Q9H633': ['R-HSA-6791226'], 'Q8TD47': ['R-HSA-9754678', 'R-HSA-9735869'], 'P22090': ['R-HSA-9754678', 'R-HSA-9735869'], 'P62701': ['R-HSA-9754678', 'R-HSA-9735869'], 'Q9BUL9': ['R-HSA-6791226'], 'P62841': ['R-HSA-9754678', 'R-HSA-9735869'], 'P62244': ['R-HSA-9754678', 'R-HSA-9735869'], 'P62263': ['R-HSA-9754678', 'R-HSA-9735869'], 'P15880': ['R-HSA-9754678', 

In [9]:
reactome_proteins = pd.read_csv("../../Results/PathwayComembership/06_ScalingReactome/Reactome_ProteinList.csv", sep=",", header=0)
list_proteins = list()
for index, row in reactome_proteins.iterrows():
    list_proteins.append(row[0])
print(len(list_proteins))

/tmp/ipykernel_447429/2391480451.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  list_proteins.append(row[0])


12102


In [10]:
dico_prot_reactome_ancestors = dict()
for prot in list_proteins:
    if prot in dico_most_precise_parent.keys():
        most_precise_parents_entity = dico_most_precise_parent[prot]
        ancestors_pathways = get_pathways_and_ancestor(most_precise_parents_entity, dico_ancestors)
        dico_prot_reactome_ancestors[prot] = ancestors_pathways
    else:
        dico_prot_reactome_ancestors[prot] = []
print(dico_prot_reactome_ancestors)
print(len(dico_prot_reactome_ancestors))

{'O95340': {'R-HSA-5668914', 'R-HSA-ROOT', 'R-HSA-3560782', 'R-HSA-3560796', 'R-HSA-3781865', 'R-HSA-1643685'}, 'P16152': {'R-HSA-2162123', 'R-HSA-ROOT', 'R-HSA-8978868', 'R-HSA-1430728', 'R-HSA-556833', 'R-HSA-2142753'}, 'P11161': {'R-HSA-187037', 'R-HSA-9006934', 'R-HSA-ROOT', 'R-HSA-9031628', 'R-HSA-162582', 'R-HSA-166520', 'R-HSA-198725'}, 'O94911': {'R-HSA-382551', 'R-HSA-ROOT', 'R-HSA-382556'}, 'Q8NCI6': {'R-HSA-1638074', 'R-HSA-ROOT', 'R-HSA-1430728', 'R-HSA-556833', 'R-HSA-9840310', 'R-HSA-428157', 'R-HSA-2022857', 'R-HSA-1630316', 'R-HSA-71387', 'R-HSA-1660662', 'R-HSA-2024101', 'R-HSA-1793185'}, 'A4D126': {'R-HSA-392499', 'R-HSA-9939291', 'R-HSA-ROOT', 'R-HSA-597592', 'R-HSA-5173105', 'R-HSA-8931838'}, 'P33897': {'R-HSA-556833', 'R-HSA-2046104', 'R-HSA-8978868', 'R-HSA-ROOT', 'R-HSA-1430728', 'R-HSA-390918', 'R-HSA-2046106', 'R-HSA-2046105', 'R-HSA-390247'}, 'Q5QJU3': {'R-HSA-ROOT', 'R-HSA-1430728', 'R-HSA-556833', 'R-HSA-428157', 'R-HSA-9845614'}, 'Q9Y6M1': {'R-HSA-428359', 

In [11]:
print(dico_prot_reactome_ancestors)
# remove artificial root from ancestors 
dico_prot_reactome_ancestors_clean = {
    prot: [p for p in pathways if p != "R-HSA-ROOT"]
    for prot, pathways in dico_prot_reactome_ancestors.items()
}
print(dico_prot_reactome_ancestors_clean)

# print(dict_nsp_inter)
# nsp_inter_top_lookup = {}
# for key, score in dict_nsp_inter.items():
#     p1, p2 = key.split("->")
#     nsp_inter_top_lookup[(p1, p2)] = score
#     nsp_inter_top_lookup[(p2, p1)] = score
# print(nsp_inter_top_lookup)

{'O95340': {'R-HSA-5668914', 'R-HSA-ROOT', 'R-HSA-3560782', 'R-HSA-3560796', 'R-HSA-3781865', 'R-HSA-1643685'}, 'P16152': {'R-HSA-2162123', 'R-HSA-ROOT', 'R-HSA-8978868', 'R-HSA-1430728', 'R-HSA-556833', 'R-HSA-2142753'}, 'P11161': {'R-HSA-187037', 'R-HSA-9006934', 'R-HSA-ROOT', 'R-HSA-9031628', 'R-HSA-162582', 'R-HSA-166520', 'R-HSA-198725'}, 'O94911': {'R-HSA-382551', 'R-HSA-ROOT', 'R-HSA-382556'}, 'Q8NCI6': {'R-HSA-1638074', 'R-HSA-ROOT', 'R-HSA-1430728', 'R-HSA-556833', 'R-HSA-9840310', 'R-HSA-428157', 'R-HSA-2022857', 'R-HSA-1630316', 'R-HSA-71387', 'R-HSA-1660662', 'R-HSA-2024101', 'R-HSA-1793185'}, 'A4D126': {'R-HSA-392499', 'R-HSA-9939291', 'R-HSA-ROOT', 'R-HSA-597592', 'R-HSA-5173105', 'R-HSA-8931838'}, 'P33897': {'R-HSA-556833', 'R-HSA-2046104', 'R-HSA-8978868', 'R-HSA-ROOT', 'R-HSA-1430728', 'R-HSA-390918', 'R-HSA-2046106', 'R-HSA-2046105', 'R-HSA-390247'}, 'Q5QJU3': {'R-HSA-ROOT', 'R-HSA-1430728', 'R-HSA-556833', 'R-HSA-428157', 'R-HSA-9845614'}, 'Q9Y6M1': {'R-HSA-428359', 

## Weighted co-membership clique without NSP

In [25]:
def load_best_scores(current_dir, path_template, label_prefix, n=29):
    """Read all n files once, canonicalize pair order, and keep only the
    max-scoring row per pair — all via vectorized pandas ops, no Python loops."""
    frames = []
    for counter in range(1, n + 1):
        path = os.path.join(current_dir, path_template.format(counter=counter))
        df = pd.read_csv(
            path, sep=",", header=0, usecols=[0, 1, 2],
            names=["p1", "p2", "score"], dtype={"score": "float64"}
        )
        # canonical unordered pair: (A,B) and (B,A) collapse to the same key
        df["pmin"] = df[["p1", "p2"]].min(axis=1)
        df["pmax"] = df[["p1", "p2"]].max(axis=1)
        df["provenance"] = f"{label_prefix}[f'{dico_top_pathway_ids[f'{counter:02d}']}']"
        frames.append(df[["pmin", "pmax", "score", "provenance"]])

    combined = pd.concat(frames, ignore_index=True)
    del frames

    # dedupe protein name strings -> small integer codes
    combined["pmin"] = combined["pmin"].astype("category")
    combined["pmax"] = combined["pmax"].astype("category")

    # keep only the max-score row per pair, in one vectorized pass
    idx = combined.groupby(["pmin", "pmax"], observed=True)["score"].idxmax()
    best = combined.loc[idx].reset_index(drop=True)
    del combined
    return best  # columns: pmin, pmax, score, provenance


mica_best = load_best_scores(
    current_dir,
    "../../Results/PathwayComembership/05_MatricesReactome/{counter:02d}_MICA_matrix_REACTOME.csv",
    "MICA",
)

# mica_best = load_best_scores(
#     current_dir,
#     "../../Results/PathwayComembership/07_TGFbeta/R-HSA-170834_MICA_matrix_REACTOME.csv",
#     "MICA",
# )

print(mica_best.head())
mica_best["score"] = mica_best["score"].fillna(0)


         pmin        pmax     score             provenance
0  A0A075B6P5  A0A075B6S6  2.105134  MICA[f'ImmuneSystem']
1  A0A075B6P5  A0A0A6YYK7  0.719057  MICA[f'ImmuneSystem']
2  A0A075B6P5  A0A0C4DH25  2.105134  MICA[f'ImmuneSystem']
3  A0A075B6P5  A0A0C4DH73  2.105134  MICA[f'ImmuneSystem']
4  A0A075B6P5  A0A0H2US87  0.994367  MICA[f'ImmuneSystem']


In [26]:
# REACTOME
df_pairs = df_pairs.copy()
df_pairs.columns = ["prot1", "prot2"] + list(df_pairs.columns[2:])
df_pairs["pmin"] = df_pairs[["prot1", "prot2"]].min(axis=1)
df_pairs["pmax"] = df_pairs[["prot1", "prot2"]].max(axis=1)

result = df_pairs.merge(
    mica_best[["pmin", "pmax", "score", "provenance"]],
    on=["pmin", "pmax"], how="left",
)
result["score"] = result["score"].fillna(0)
result["provenance"] = result["provenance"].fillna("none")

reactome_weighted_clique = result[["prot1", "prot2", "score", "provenance"]]
reactome_weighted_clique.rename(columns={'score':'comembershipScore'}, inplace=True)
reactome_weighted_clique.sort_values(by="comembershipScore", ascending=False)

/tmp/ipykernel_447429/3281238369.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reactome_weighted_clique.rename(columns={'score':'comembershipScore'}, inplace=True)


,prot1,prot2,comembershipScore,provenance
37610189,P53985,P35613,3.781827,MICA[f'Disease']
21846934,Q99259,Q05329,3.781827,MICA[f'NeuronalSystem']
46483373,Q9BVG9,P48651,3.781827,MICA[f'Metabolism']
65850384,Q96E22,Q86SQ9,3.781827,MICA[f'Disease']
67182928,Q9HBL8,P00966,3.781827,MICA[f'Disease']
...,...,...,...,...
7010549,Q9H270,P14784,0.000000,none
32338962,Q06055,Q8NHB7,0.000000,none
32338963,Q06055,Q8N8S7,0.000000,none
32338964,Q06055,Q6VUC0,0.000000,none


In [27]:
reactome_weighted_clique.to_csv(
    "../../Results/PathwayComembership/06_ScalingReactome/Reactome96WeightedComembershipClique_NO_NSP.csv",
    sep=",", header=True, index=False,
)
print(reactome_weighted_clique.head())
print(len(reactome_weighted_clique))

    prot1   prot2  comembershipScore           provenance
0  O95340  P16152           0.737287  MICA[f'Metabolism']
1  O95340  P11161           0.000000                 none
2  O95340  O94911           0.000000                 none
3  O95340  Q8NCI6           0.737287  MICA[f'Metabolism']
4  O95340  A4D126           0.000000                 none
73223151


In [28]:
reactome_weighted_clique.to_csv(
    "../../Results/PathwayComembership/06_ScalingReactome/Reactome96WeightedComembershipClique_NO_NSP.tsv",
    sep="\t", header=None, index=False,
)

In [29]:
reactome_weighted_clique.to_csv(
    "/home/cbeust/Projects/2026/TestsRWRComembership/RWRComembershipTestFromScratch/Data/ReactomeCliques/Reactome96WeightedComembershipClique_NO_NSP.tsv",
    columns=["prot1", "prot2", "comembershipScore"],
    sep="\t", header=None, index=False,
)

## Cosine similarity

In [12]:
print(dico_most_precise_parent)
print(len(dico_most_precise_parent))
print(dico_prot_reactome_ancestors_clean)
print(len(dico_prot_reactome_ancestors))
print(len(dico_prot_reactome_ancestors_clean))
print(dico_er_per_pathway)

{'P46976': ['R-HSA-5357609', 'R-HSA-3828062', 'R-HSA-3814836', 'R-HSA-3785653'], 'Q13158': ['R-HSA-2562578'], 'Q14790': ['R-HSA-2562578'], 'Q92851': ['R-HSA-6803207'], 'Q13546': ['R-HSA-2562578', 'R-HSA-937041'], 'Q7Z434': ['R-HSA-9705671', 'R-HSA-9692916'], 'Q9BYX4': ['R-HSA-9705671', 'R-HSA-9692916'], 'O95786': ['R-HSA-9705671', 'R-HSA-9692916'], 'Q9C037': ['R-HSA-9705671'], 'Q14258': ['R-HSA-9705671', 'R-HSA-9692916'], 'Q8IUD6': ['R-HSA-9705671'], 'O14920': ['R-HSA-9705671', 'R-HSA-975144', 'R-HSA-937041'], 'Q9Y6K9': ['R-HSA-9705671', 'R-HSA-975144', 'R-HSA-937041'], 'O15111': ['R-HSA-9705671', 'R-HSA-975144', 'R-HSA-937041'], 'Q9H633': ['R-HSA-6791226'], 'Q8TD47': ['R-HSA-9754678', 'R-HSA-9735869'], 'P22090': ['R-HSA-9754678', 'R-HSA-9735869'], 'P62701': ['R-HSA-9754678', 'R-HSA-9735869'], 'Q9BUL9': ['R-HSA-6791226'], 'P62841': ['R-HSA-9754678', 'R-HSA-9735869'], 'P62244': ['R-HSA-9754678', 'R-HSA-9735869'], 'P62263': ['R-HSA-9754678', 'R-HSA-9735869'], 'P15880': ['R-HSA-9754678', 

In [13]:
vector_proteins = {}

for protein in dico_parents_up:
    vector = [0] * len(list_pathways)

    for i, pathway in enumerate(list_pathways):
        if protein in dico_parents_up and pathway in dico_parents_up[protein]:
            vector[i] = 1

    vector_proteins[protein] = vector

In [14]:
def compute_cosine_sim(prot1, prot2, list_pathways, vector_proteins):
    if prot1 in vector_proteins:
        pathways_prot1 = np.array(vector_proteins[prot1])
    else:
        pathways_prot1 = np.array([0] * len(list_pathways))

    if prot2 in vector_proteins:
        pathways_prot2 = np.array(vector_proteins[prot2])
    else:
        pathways_prot2 = np.array([0] * len(list_pathways))

    num = np.dot(pathways_prot1, pathways_prot2)
    norm_prot1_vec = np.linalg.norm(pathways_prot1)
    norm_prot2_vec = np.linalg.norm(pathways_prot2)

    if norm_prot1_vec == 0 or norm_prot2_vec == 0:
        return 0.0

    return num / (norm_prot1_vec * norm_prot2_vec)


print(sum(vector_proteins["P62987"]))
print(sum(vector_proteins["P62979"]))
print(compute_cosine_sim("P62987", "P62979", list_pathways, vector_proteins))

429
436
0.9919399952257395


In [15]:
# REACTOME 
comembership_clique = pd.read_csv("../../Results/PathwayComembership/06_ScalingReactome/Reactome_ComembershipClique.csv")
df_weight_cos = pd.DataFrame(columns=["prot1", "prot2", "scoreCosine"])
results = []
for row in comembership_clique.itertuples(index=False):
    weight = compute_cosine_sim(row[0], row[1], list_pathways, vector_proteins)
    results.append((row[0], row[1], weight))

df_weight_cos = pd.DataFrame(results, columns=["prot1", "prot2", "scoreCosine"])
df_weight_cos.to_csv("../../Results/PathwayComembership/06_ScalingReactome/Reactome96WeightedComembershipClique_Cosine.csv",
    sep=",", header=True, index=False)
df_weight_cos.to_csv("../../Results/PathwayComembership/06_ScalingReactome/Reactome96WeightedComembershipClique_Cosine.tsv",
    sep="\t", header=None, index=False)

In [16]:
df_weight_cos.to_csv("/home/cbeust/Projects/2026/TestsRWRComembership/RWRComembershipTestFromScratch/Data/ReactomeCliques/Reactome96WeightedComembershipClique_Cosine.tsv",
    sep="\t", header=None, index=False)

## Jaccard index

In [17]:
def compute_jaccard_sim(prot1, prot2, list_pathways, vector_proteins):
    if prot1 in vector_proteins:
        pathways_prot1 = np.array(vector_proteins[prot1]) > 0 
    else:
        pathways_prot1 = np.array([0] * len(list_pathways))

    if prot2 in vector_proteins:
        pathways_prot2 = np.array(vector_proteins[prot2]) > 0
    else:
        pathways_prot2 = np.array([0] * len(list_pathways))

    intersection = np.logical_and(pathways_prot1, pathways_prot2).sum()
    union = np.logical_or(pathways_prot1, pathways_prot2).sum()

    if union == 0:
        return 0.0
    return intersection / union

print(compute_jaccard_sim("P62987", "P62979", list_pathways, vector_proteins))

0.9839449541284404


In [18]:
# REACTOME
comembership_clique = pd.read_csv("../../Results/PathwayComembership/06_ScalingReactome/Reactome_ComembershipClique.csv")
results = []
for row in comembership_clique.itertuples(index=False):
    weight = compute_jaccard_sim(row[0], row[1], list_pathways, vector_proteins)
    results.append((row[0], row[1], weight))

df_weight_jac = pd.DataFrame(results, columns=["prot1", "prot2", "scoreJaccard"])
df_weight_jac.to_csv("../../Results/PathwayComembership/06_ScalingReactome/Reactome96WeightedComembershipClique_Jaccard.csv",
    sep=",", header=True, index=False)
df_weight_jac.to_csv("../../Results/PathwayComembership/06_ScalingReactome/Reactome96WeightedComembershipClique_Jaccard.tsv",
    sep="\t", header=None, index=False)

In [19]:
df_weight_jac.to_csv("/home/cbeust/Projects/2026/TestsRWRComembership/RWRComembershipTestFromScratch/Data/ReactomeCliques/Reactome96WeightedComembershipClique_Jaccard.tsv",
    sep="\t", header=None, index=False)